# Random Forest - Bagging Ensemble
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajit-ai/Data_Science/blob/main/03_Machine_Learning/algorithms/random_forest.ipynb)

A Random Forest trains many decision trees on bootstrap samples with random feature subsets and averages their votes. This reduces variance dramatically compared to a single tree.

**Covered:** single tree vs forest, OOB score, feature importance, n_estimators effect.

## 1. Data: breast cancer diagnosis

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

data = load_breast_cancer()
Xtr, Xte, ytr, yte = train_test_split(
    data.data, data.target, test_size=0.25, random_state=42, stratify=data.target)

single = DecisionTreeClassifier(random_state=42).fit(Xtr, ytr)
forest = RandomForestClassifier(n_estimators=200, oob_score=True,
                                random_state=42, n_jobs=-1).fit(Xtr, ytr)

print(f"single tree : {single.score(Xte, yte):.4f}")
print(f"random forest: {forest.score(Xte, yte):.4f}")
print(f"OOB score   : {forest.oob_score_:.4f}")

Out-of-bag (OOB) score is free validation: each tree is evaluated on the ~37% samples it never saw during bootstrapping.

## 2. How many trees do we need?

In [ ]:
import matplotlib.pyplot as plt
sizes = [1, 5, 10, 25, 50, 100, 200, 400]
scores = []
for n in sizes:
    rf = RandomForestClassifier(n_estimators=n, random_state=42,
                                n_jobs=-1).fit(Xtr, ytr)
    scores.append(rf.score(Xte, yte))
plt.plot(sizes, scores, "o-")
plt.xscale("log"); plt.xlabel("n_estimators"); plt.ylabel("test accuracy")
plt.title("Accuracy saturates - more trees never hurt accuracy, only speed"); plt.show()

## 3. Feature importance + confusion matrix

In [ ]:
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay

imp = pd.Series(forest.feature_importances_, index=data.feature_names)
imp.nlargest(10).sort_values().plot(kind="barh", title="Top-10 importances")
plt.tight_layout(); plt.show()

ConfusionMatrixDisplay.from_estimator(forest, Xte, yte, display_labels=data.target_names)
plt.title("Random Forest"); plt.show()

## Bagging vs Boosting
| | Random Forest (bagging) | Boosting (XGBoost etc.) |
|---|---|---|
| Trees trained | in parallel, independent | sequentially, each fixes previous errors |
| Goal | reduce **variance** | reduce **bias** |
| Overfits when | trees too deep | too many rounds / lr too high |